# Sequencing depth is a stage covariate in Task 1

This notebook reproduces, from the released Task 1 files only, three facts:

1. **E10.5 cells are sequenced ~40 % shallower than E8.5/E9.5 cells** (median UMI per cell 10.9k / 11.1k / 6.7k).
2. **The drop is within cell type**, so it is a property of the library, not of the changing composition.
3. **A model that fits the E9.5 → E10.5 change and continues it inherits a depth signature.** Thinning E9.5 to E10.5 depth *before* measuring the step removes about a third of the apparent pseudobulk DE calls and halves the correlation between Δ and expression level.

Runtime: ~1 minute, ~9 GB RAM (three 32,285-gene matrices held as counts). Set `VEC_DATA` to the folder that holds `E8.5_RNA.h5ad`, `E9.5_RNA.h5ad`, `E10.5_RNA.h5ad` (E10.5 is the P2 validation stage, released to opted-in teams in P3; section 1 runs without it).

Data credit: Dr Qingquan Zhang and Dr Neil Chi (UCSD), Virtual Embryo Challenge release. Nothing here reads a held-out test stage.

In [1]:
import os, numpy as np, pandas as pd, scipy.sparse as sp, anndata as ad
from scipy.stats import spearmanr
DATA = os.environ.get("VEC_DATA", "./data")
FILES = {"E8.5": "E8.5_RNA.h5ad", "E9.5": "E9.5_RNA.h5ad", "E10.5": "E10.5_RNA.h5ad"}
FILES = {s: os.path.join(DATA, f) for s, f in FILES.items() if os.path.exists(os.path.join(DATA, f))}
print("stages found:", list(FILES))

stages found: ['E8.5', 'E9.5', 'E10.5']


## 1. Recover integer counts
`X` is log1p(CP10K). Per cell, the smallest non-zero value corresponds to one UMI, so `expm1(x) / expm1(min nonzero)` is an integer count. The same idea is published as *Count Recovery* (I. Habib, community resources); a 12-line version is kept here so the notebook is self-contained. The maximum rounding error is printed as a check (it should be ≪ 0.5).

In [2]:
def recover_counts(Xlog):
    M = sp.csr_matrix(Xlog, dtype=np.float64, copy=True)
    M.data = np.expm1(M.data); M.data[M.data < 1e-9] = 0; M.eliminate_zeros()
    nnz = np.diff(M.indptr); assert (nnz > 0).all(), "empty cell"
    unit = np.minimum.reduceat(M.data, M.indptr[:-1])
    c = M.data / unit[np.repeat(np.arange(M.shape[0]), nnz)]
    err = float(np.abs(c - np.round(c)).max()); M.data = np.round(c)
    return M, err

def to_log(C, target=1e4):
    C = sp.csr_matrix(C); s = np.asarray(C.sum(1)).ravel()
    L = sp.csr_matrix(sp.diags(target / np.maximum(s, 1e-12)) @ C, dtype=np.float64); L.data = np.log1p(L.data); return L

D = {}
for stage, path in FILES.items():
    A = ad.read_h5ad(path); Cnt, err = recover_counts(A.X)
    D[stage] = dict(C=Cnt, lab=A.obs["celltype"].astype(str).values, genes=A.var_names.values,
                    umi=np.asarray(Cnt.sum(1)).ravel(), ngenes=np.diff(Cnt.indptr), err=err)
genes = D["E9.5"]["genes"]; assert all((D[s]["genes"] == genes).all() for s in D)
stage_tab = pd.DataFrame([{"stage": s, "cells": len(d["umi"]), "median_UMI": int(np.median(d["umi"])), "mean_UMI": int(d["umi"].mean()),
                           "total_UMI_M": round(d["umi"].sum() / 1e6, 1), "median_genes_detected": int(np.median(d["ngenes"])),
                           "count_recovery_max_err": round(d["err"], 4)} for s, d in D.items()])
stage_tab

,stage,cells,median_UMI,mean_UMI,total_UMI_M,median_genes_detected,count_recovery_max_err
0,E8.5,16787,10857,12224,205.2,3836,0.0003
1,E9.5,17057,11080,12698,216.6,3878,0.0005
2,E10.5,24979,6723,8052,201.1,2869,0.0012


**Reading.** Total UMI per stage is nearly constant (~200–217 M) while E10.5 has 1.5× the cells, so each E10.5 cell carries ~0.6× the counts and detects ~26 % fewer genes. Anyone forecasting E12.5 should note the same arithmetic: the board contract's E12.5 cell count (~12.8k, from truth-half 638 × 2 × 10) is about half of E10.5's, so E12.5 may be *deeper* per cell than E10.5, not shallower.

## 2. The drop is within cell type
If the depth change were composition (new, smaller cell types at E10.5), matched labels would keep their depth. They do not: 10 of 12 shared labels with ≥ 50 cells at both stages sit at 0.49–0.70.

In [3]:
if "E10.5" in D:
    rows = []
    for s in D:
        df = pd.DataFrame({"ct": D[s]["lab"], "umi": D[s]["umi"]}); g = df.groupby("ct")["umi"].agg(n="size", med="median"); g["stage"] = s; rows.append(g.reset_index())
    ct = pd.concat(rows); med = ct.pivot(index="ct", columns="stage", values="med"); n = ct.pivot(index="ct", columns="stage", values="n")
    keep = med.index[(n["E9.5"].fillna(0) >= 50) & (n["E10.5"].fillna(0) >= 50)]
    per_ct = med.loc[keep].round(0); per_ct["ratio_E10.5/E9.5"] = (med.loc[keep, "E10.5"] / med.loc[keep, "E9.5"]).round(3)
    display(per_ct.sort_values("ratio_E10.5/E9.5"))
    print("median within-type ratio:", per_ct["ratio_E10.5/E9.5"].median())

median within-type ratio: 0.588


stage,E10.5,E8.5,E9.5,ratio_E10.5/E9.5
ct,,,,
aPHM,4056.0,NaN,8258.0,0.491
pPHM,4207.0,NaN,8500.0,0.495
pSHF,5276.0,9056.0,9569.0,0.551
Pericardium,4610.0,10274.0,8352.0,0.552
ST,4777.0,NaN,8552.0,0.559
Surface Ectoderm,4987.0,9925.0,8692.0,0.574
IFT-CM,8868.0,12134.0,14736.0,0.602
Blood,3608.0,8412.0,5803.0,0.622
BEC,8126.0,NaN,12127.0,0.670


## 3. What depth does to the measured stage-to-stage change
Pseudobulk the log-normalised matrices (the quantity the DE metric and most "shift" baselines operate on) and look at the step Δ = E10.5 − E9.5 as a function of expression level.

Then build a **pure-depth control**: binomially thin the E9.5 counts to the E10.5 median depth (ratio 0.607), renormalise, and measure Δ = E9.5-thinned − E9.5. No biology changes; only depth does.

Finally measure the **depth-neutral step** Δ = E10.5 − E9.5-thinned.

In [4]:
def thin(C, ratio, seed=0):
    rng = np.random.default_rng(seed); T = sp.csr_matrix(C, copy=True)
    T.data = rng.binomial(T.data.astype(np.int64), ratio).astype(np.float64); T.eliminate_zeros(); return T

def step_stats(pb_from, pb_to, mask):
    d = pb_to - pb_from; lvl = 0.5 * (pb_from + pb_to)
    return dict(spearman_rho_delta_vs_level=round(float(spearmanr(d[mask], lvl[mask]).correlation), 3),
                n_genes_abs_delta_ge_0_25=int((np.abs(d[mask]) >= 0.25).sum()),
                n_down=int((d[mask] <= -0.25).sum()), n_up=int((d[mask] >= 0.25).sum()),
                mean_delta_top200_expressed=round(float(d[np.argsort(-lvl)[:200]].mean()), 3))

if "E10.5" in D:
    PB = {s: np.asarray(to_log(D[s]["C"]).mean(0)).ravel() for s in D}
    ratio = float(np.median(D["E10.5"]["umi"]) / np.median(D["E9.5"]["umi"])); print("depth ratio E10.5 / E9.5 =", round(ratio, 3))
    PB["E9.5_thinned"] = np.asarray(to_log(thin(D["E9.5"]["C"], ratio)).mean(0)).ravel()
    expressed = (PB["E9.5"] > 0.05) | (PB["E10.5"] > 0.05); print("genes with pseudobulk > 0.05 in either stage:", int(expressed.sum()))
    res = {"E8.5 -> E9.5 (measured; depth ratio 1.02)": step_stats(PB["E8.5"], PB["E9.5"], (PB["E8.5"] > 0.05) | (PB["E9.5"] > 0.05)),
           "E9.5 -> E10.5 (measured)": step_stats(PB["E9.5"], PB["E10.5"], expressed),
           "E9.5 -> E9.5 thinned to 0.607 (pure depth, no biology)": step_stats(PB["E9.5"], PB["E9.5_thinned"], expressed),
           "E9.5 thinned -> E10.5 (depth-neutral step)": step_stats(PB["E9.5_thinned"], PB["E10.5"], expressed)}
    res = pd.DataFrame(res).T; display(res)
    d_meas = PB["E10.5"] - PB["E9.5"]; d_depth = PB["E9.5_thinned"] - PB["E9.5"]
    print("Spearman(measured Δ, pure-depth Δ):", round(float(spearmanr(d_meas[expressed], d_depth[expressed]).correlation), 3))
    top = np.argsort(-np.abs(d_meas * expressed))[:500]
    print("sign agreement of pure-depth Δ with the 500 largest measured |Δ|:", round(float((np.sign(d_meas[top]) == np.sign(d_depth[top])).mean()), 3))

depth ratio E10.5 / E9.5 = 0.607
genes with pseudobulk > 0.05 in either stage: 10603
Spearman(measured Δ, pure-depth Δ): 0.5
sign agreement of pure-depth Δ with the 500 largest measured |Δ|: 0.844


,spearman_rho_delta_vs_level,n_genes_abs_delta_ge_0_25,n_down,n_up,mean_delta_top200_expressed
E8.5 -> E9.5 (measured; depth ratio 1.02),-0.092,55.0,28.0,27.0,-0.043
E9.5 -> E10.5 (measured),-0.414,462.0,390.0,72.0,-0.213
"E9.5 -> E9.5 thinned to 0.607 (pure depth, no biology)",-0.950,0.0,0.0,0.0,-0.035
E9.5 thinned -> E10.5 (depth-neutral step),-0.227,314.0,221.0,93.0,-0.179


**Reading the table.**
- The step between two stages of *equal* depth (E8.5 → E9.5) has almost no level correlation (ρ ≈ −0.09) and 55 pseudobulk calls at |Δ| ≥ 0.25.
- The E9.5 → E10.5 step has ρ = −0.41 and 462 calls, 390 of them down. Pure thinning reproduces the *correlation* almost perfectly (ρ = −0.95) with small per-gene magnitude (no gene reaches 0.25; top-200 mean −0.035): depth is a consistent, small, level-dependent downward push on thousands of genes at once.
- Measuring the step from depth-matched E9.5 removes 148 of the 462 calls (−32 %), 169 of the 390 down-calls, and halves ρ. **What remains (314 calls, ρ = −0.23) is biology**: E9.5 and E10.5 share only 13 labels, so most of the real change is compositional.

Two consequences for modelling:
1. A model that fits Δ(E9.5 → E10.5) per cell type and *continues* it to E12.5 (any "shift", "velocity" or "transport" recipe) propagates the depth component as if it were development. On our P3 shot 1 this showed up as a gene-gene co-variation skill (CSS) of 63.9 against ~69 for E10.5 cells submitted verbatim (public T1:test board, team Second Heart Field).
2. Correcting the symptom afterwards (rescaling abundant genes, de-trending Δ against level, replacing "frozen" genes) does not recover the points. Five such families lost 1.3–2.5 headline points each on T1:val (`failed_corrections.md`); per-cell renormalisation with cell-dependent factors also moves every cell in the PC basis the MMD metric is fitted in. Thin the *earlier* stage to the donor's depth before fitting the step instead (`depth_neutral_fit.md`).

## 4. Save the small tables
These are what the figure in the README is drawn from.

In [5]:
os.makedirs("data", exist_ok=True)
stage_tab.to_csv("data/stage_depth.csv", index=False)
if "E10.5" in D:
    per_ct.reset_index().to_csv("data/per_celltype_depth.csv", index=False)
    res.to_csv("data/step_statistics.csv")
    pd.DataFrame({"gene": genes, **{f"pb_{k}": v for k, v in PB.items()}, "expressed": expressed}).to_csv("data/pseudobulk_log1pCP10K.csv.gz", index=False)
    pd.concat([pd.DataFrame({"stage": s, "celltype": D[s]["lab"], "umi": D[s]["umi"].astype(int), "genes_detected": D[s]["ngenes"]}) for s in D]).to_csv("data/per_cell_depth.csv.gz", index=False)
print("written")

written
